In [1]:
import unicodedata
import tkinter as tk

# --- Bảng luật ---
RULES_3 = {
    "ngh": "e01f",
}

RULES_2 = {
    "ph": "e021",
    "th": "e022",
    "gi": "e013",
    "tr": "e01d",
    "ch": "e01d",
    "nh": "e020",
    "ng": "e01f",
    "kh": "e01e",
    "gh": "e015",
    "ia": "e00a",
    "uô": "e00c",
    "uơ": "e00c",
    "uâ": "e00c",
    "ươ": "e00b",
    "yê": "e00a",
    "iê": "e00a",
}

RULES_1 = {
    "a": "e000",
    "ă": "e001",
    "â": "e002",
    "b": "e011",
    "c": "e012",
    "d": "e013",
    "đ": "e014",
    "e": "e003",
    "ê": "e004",
    "g": "e015",
    "h": "e016",
    "i": "e005",
    "k": "e012",
    "l": "e017",
    "m": "e018",
    "n": "e019",
    "o": "e006",
    "ô": "e007",
    "ơ": "e002",
    "p": "e011",
    "q": "e012",
    "r": "e013",
    "s": "e01c",
    "t": "e01a",
    "u": "e008",
    "ư": "e009",
    "v": "e01b",
    "x": "e01c",
    "y": "e005",
}

FINAL_CONSONANTS_2 = {"ng": "e015", "nh": "e015"}
FINAL_CONSONANTS_1 = {"m": "e024", "n": "e025", "t": "e026"}

FINAL_VOWELS = {
    "o": "e00f",
    "u": "e00f",
    "i": "e00e",
    "y": "e00e",
}

ONSETS_3 = ["ngh"]

ONSETS_2 = [
    "ph", "th", "tr", "ch", "nh", "ng", "kh", "gh", "gi"
]

ONSETS_1 = list("bcdđghklmnpqrstvx")


# ============ BỎ DẤU THANH (GIỮ Â, Ă, Ê, Ơ, Ư, Đ) ============
def remove_tone_marks(text):
    TONE_MARKS = {
        "\u0300",  # huyền
        "\u0301",  # sắc
        "\u0303",  # ngã
        "\u0309",  # hỏi
        "\u0323",  # nặng
    }

    normalized = unicodedata.normalize("NFD", text)
    result = []

    for ch in normalized:
        if ch in TONE_MARKS:
            continue
        result.append(ch)

    return unicodedata.normalize("NFC", "".join(result))
# ============ THÊM O VÀO TIẾNG THIẾU PHỤ ÂM ĐẦU ============

SPECIAL_ONSET_INSERT = chr(int("e006", 16))

def insert_virtual_onset(word):
    if not word:
        return word

    # Nếu đã có phụ âm đầu thật thì thôi
    onset, rime = split_onset_rime(word)
    if onset:
        return word

    # Nếu bắt đầu bằng o hoặc u thì không thêm
    if rime.startswith(chr(int("e00f", 16))):
        return word

    return SPECIAL_ONSET_INSERT + word

def apply_virtual_onset_rule(text):
    words = text.split()
    words = [insert_virtual_onset(w) for w in words]
    return " ".join(words)

# ============ CHUYỂN MỘT SỐ NGUYÊN ÂM ĐẶC BIỆT LÊN ĐẦU ============

PROTECTED_CLUSTERS = ["iê", "yê", "ia", "uô", "uơ"]
VOWELS_TO_MOVE = ["ươ", "ô", "ê", "ơ", "e", "â"]

def reorder_syllable(syllable):
    # Nếu chứa cụm nguyên âm bảo vệ → không đụng vào phần đó
    protected_positions = []

    for cluster in PROTECTED_CLUSTERS:
        start = syllable.find(cluster)
        if start != -1:
            protected_positions.extend(range(start, start + len(cluster)))

    # Tìm nguyên âm cần đảo nhưng không nằm trong vùng bảo vệ
    for v in VOWELS_TO_MOVE:
        idx = syllable.find(v)
        if idx != -1 and idx not in protected_positions:
            syllable = syllable[:idx] + syllable[idx+len(v):]
            return v + syllable

    return syllable

def apply_reorder_rule(text):
    syllables = text.split()
    syllables = [reorder_syllable(s) for s in syllables]
    return " ".join(syllables)


# ============ ÁP DỤNG KHOÁ ĐUÔI ============

def split_onset_rime(word):
    for o in ONSETS_3:
        if word.startswith(o):
            return o, word[len(o):]

    for o in ONSETS_2:
        if word.startswith(o):
            return o, word[len(o):]

    if word and word[0] in ONSETS_1:
        return word[0], word[1:]

    return "", word  # không có phụ âm đầu

def get_protected_positions(rime):
    protected = set()
    for cluster in PROTECTED_CLUSTERS:
        start = rime.find(cluster)
        if start != -1:
            for i in range(start, start + len(cluster)):
                protected.add(i)
    return protected


def transform_rime(rime):
    protected_pos = get_protected_positions(rime)

    # Âm tiết đóng → chỉ đổi phụ âm cuối
    if len(rime) >= 2 and rime[-2:] in FINAL_CONSONANTS_2:
        return rime[:-2] + chr(int(FINAL_CONSONANTS_2[rime[-2:]], 16))

    if rime and rime[-1] in FINAL_CONSONANTS_1:
        return rime[:-1] + chr(int(FINAL_CONSONANTS_1[rime[-1]], 16))

    # Âm tiết mở → đổi nguyên âm không bị bảo vệ
    result = ""
    for i, ch in enumerate(rime):
        if i in protected_pos:
            result += ch
        elif ch in FINAL_VOWELS:
            result += chr(int(FINAL_VOWELS[ch], 16))
        else:
            result += ch

    return result


def apply_final_rules_to_text(text):
    words = text.split()
    new_words = []

    for w in words:
        onset, rime = split_onset_rime(w)
        rime = transform_rime(rime)
        new_words.append(onset + rime)

    return " ".join(new_words)


# ================= HÀM CHUYỂN TỰ =================

def encode_custom(text):
    text = text.lower()
    text = remove_tone_marks(text)

    text = apply_final_rules_to_text(text)
    text = apply_virtual_onset_rule(text)
    text = apply_reorder_rule(text)


    i = 0
    output = ""

    while i < len(text):
        # ưu tiên cụm 3 ký tự
        if i + 3 <= len(text) and text[i:i+3] in RULES_3:
            code = RULES_3[text[i:i+3]]
            output += chr(int(code, 16))
            i += 3
            continue

        # cụm 2 ký tự
        if i + 2 <= len(text) and text[i:i+2] in RULES_2:
            code = RULES_2[text[i:i+2]]
            output += chr(int(code, 16))
            i += 2
            continue

        # 1 ký tự
        ch = text[i]
        if ch in RULES_1:
            code = RULES_1[ch]
            output += chr(int(code, 16))
        else:
            output += ch  # giữ nguyên ký tự lạ (dấu cách, số, ...)
        i += 1

    return output


In [2]:
# ================= GUI =================

def on_text_change(event=None):
    encoded = encode_custom(entry.get())
    output_label.config(text=encoded)

root = tk.Tk()
root.title("PUA Font Encoder")

entry = tk.Entry(root, font=("Segoe UI", 14), width=40)
entry.pack(padx=20, pady=10)
entry.bind("<KeyRelease>", on_text_change)

output_label = tk.Label(
    root,
    text="",
    font=("Khoa Dau", 48),  # font PUA của bạn
    bg="white"
)
output_label.pack(padx=20, pady=20, fill="both")

on_text_change()
root.mainloop()